# 06 — Build monthly OSM intrinsic temporal indicators for 895 Petrópolis cells

This notebook extends the validated production pipeline from static/full-period metrics to a **monthly spatial-temporal panel** for the 895 complete 1 km IBGE cells intersecting Petrópolis.

The panel preserves `grid_1km_id` as the spatial key and separates two temporal semantics:

- **state variables**: road length and road-element count observed at monthly snapshots;
- **activity variables**: unique contributors and contribution counts during monthly intervals.

Contribution-type counts (`creation`, `deletion`, `tagChange`, `geometryChange`) are retained as potentially overlapping activity indicators. They must **not** be interpreted as mutually exclusive components whose sum necessarily equals total contributions.

Only complete calendar months contained within the current OSHDB temporal extent are queried. Raw API responses are checkpointed by metric and batch so interrupted runs can resume without repeating validated requests.

In [1]:
from pathlib import Path
import json
import math
import time
from datetime import datetime, timezone

import pandas as pd
import geopandas as gpd
import requests

API_BASE = 'https://api.ohsome.org/v1'
ROAD_FILTER = 'highway=* and geometry:line'
PERIOD = 'P1M'
BATCH_SIZE = 25
REQUEST_TIMEOUT = 300
MAX_ATTEMPTS = 5
TRANSIENT_STATUS = {429, 500, 502, 503, 504}

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, start.parent, *start.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'Petropolis').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root.')

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_DIR = PROJECT_ROOT / 'data' / 'intermediate' / 'ohsome' / 'Petropolis'
FULL_BOUNDARIES_FILE = INPUT_DIR / 'petropolis_grid_1km_ohsome_boundaries.geojson'
GRID_FILE = PROJECT_ROOT / 'data' / 'processed' / 'Petropolis' / 'petropolis_grid_1km.parquet'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ohsome' / 'Petropolis' / 'monthly_timeseries_895_cells'
RESULTS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'Petropolis'
OUTPUT_PANEL = RESULTS_DIR / 'petropolis_ohsome_monthly_timeseries_895_cells.parquet'
OUTPUT_SUMMARY = RESULTS_DIR / 'petropolis_ohsome_temporal_summary_895_cells.parquet'
OUTPUT_SUMMARY_GEO = RESULTS_DIR / 'petropolis_grid_1km_with_temporal_summary.parquet'
MANIFEST_FILE = RAW_DIR / 'manifest.json'

for path in (FULL_BOUNDARIES_FILE, GRID_FILE):
    if not path.exists():
        raise FileNotFoundError(f'Required upstream output not found: {path}')
RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({'User-Agent':'sotm-br-2026-research/0.6 (academic OSM analysis)'})

## 1. Read ohsome metadata and define complete calendar months

In [2]:
response = session.get(f'{API_BASE}/metadata', timeout=REQUEST_TIMEOUT)
response.raise_for_status()
metadata = response.json()
(RAW_DIR / 'ohsome_metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

api_version = metadata.get('apiVersion')
temporal = metadata.get('extractRegion', {}).get('temporalExtent', {})
oshdb_start = pd.Timestamp(temporal.get('fromTimestamp'))
oshdb_end = pd.Timestamp(temporal.get('toTimestamp'))
if pd.isna(oshdb_start) or pd.isna(oshdb_end):
    raise ValueError('Could not read extractRegion.temporalExtent from /metadata.')

series_start = (oshdb_start + pd.offsets.MonthBegin(1)).normalize()
series_end = pd.Timestamp(year=oshdb_end.year, month=oshdb_end.month, day=1, tz='UTC')
if series_end <= series_start:
    raise ValueError('OSHDB extent is too short to construct complete monthly intervals.')

time_spec = f"{series_start.strftime('%Y-%m-%d')}/{series_end.strftime('%Y-%m-%d')}/{PERIOD}"
expected_interval_starts = pd.date_range(series_start, series_end, freq='MS', inclusive='left', tz='UTC')
expected_snapshot_times = pd.date_range(series_start, series_end, freq='MS', inclusive='both', tz='UTC')

print('API version:', api_version)
print('OSHDB temporal extent:', oshdb_start, '→', oshdb_end)
print('Complete-month query:', time_spec)
print('Expected monthly activity intervals:', len(expected_interval_starts))
print('Expected state snapshots:', len(expected_snapshot_times))

API version: 1.10.4
OSHDB temporal extent: 2007-10-08 00:00:00+00:00 → 2026-07-27 09:00:00+00:00
Complete-month query: 2007-11-01/2026-07-01/P1M
Expected monthly activity intervals: 224
Expected state snapshots: 225


## 2. Validate the 895 boundaries and build deterministic batches

In [3]:
grid = gpd.read_parquet(GRID_FILE)
feature_collection = json.loads(FULL_BOUNDARIES_FILE.read_text(encoding='utf-8'))
features = feature_collection.get('features', [])
feature_by_id = {str(f.get('properties', {}).get('id')): f for f in features}
grid_ids = grid['grid_1km_id'].astype(str).tolist()
boundary_ids = list(feature_by_id)

boundary_checks = {
    'grid_rows_895': len(grid) == 895,
    'boundaries_895': len(features) == 895,
    'grid_ids_unique': grid['grid_1km_id'].is_unique,
    'boundary_ids_unique': len(boundary_ids) == len(set(boundary_ids)),
    'ids_match': set(grid_ids) == set(boundary_ids),
}
print(boundary_checks)
if not all(boundary_checks.values()):
    raise ValueError(f'Boundary validation failed: {boundary_checks}')

ordered_ids = sorted(grid_ids)
batches = [ordered_ids[i:i+BATCH_SIZE] for i in range(0, len(ordered_ids), BATCH_SIZE)]
print(f'{len(batches)} deterministic batches; standard size={BATCH_SIZE}; last size={len(batches[-1])}')

def batch_feature_collection(ids):
    return {'type':'FeatureCollection', 'features':[feature_by_id[i] for i in ids]}

{'grid_rows_895': True, 'boundaries_895': True, 'grid_ids_unique': True, 'boundary_ids_unique': True, 'ids_match': True}
36 deterministic batches; standard size=25; last size=20


## 3. Define temporal metrics

In [4]:
METRICS = {
    'road_length_m': {
        'endpoint':'/elements/length/groupBy/boundary', 'kind':'snapshot', 'extra':{}
    },
    'road_element_count': {
        'endpoint':'/elements/count/groupBy/boundary', 'kind':'snapshot', 'extra':{}
    },
    'contributors_count': {
        'endpoint':'/users/count/groupBy/boundary', 'kind':'interval', 'extra':{}
    },
    'contributions_count': {
        'endpoint':'/contributions/count/groupBy/boundary', 'kind':'interval', 'extra':{}
    },
    'creation_contributions_count': {
        'endpoint':'/contributions/count/groupBy/boundary', 'kind':'interval', 'extra':{'contributionType':'creation'}
    },
    'deletion_contributions_count': {
        'endpoint':'/contributions/count/groupBy/boundary', 'kind':'interval', 'extra':{'contributionType':'deletion'}
    },
    'tag_change_contributions_count': {
        'endpoint':'/contributions/count/groupBy/boundary', 'kind':'interval', 'extra':{'contributionType':'tagChange'}
    },
    'geometry_change_contributions_count': {
        'endpoint':'/contributions/count/groupBy/boundary', 'kind':'interval', 'extra':{'contributionType':'geometryChange'}
    },
}
pd.DataFrame([{'metric':k, **v} for k,v in METRICS.items()])

,metric,endpoint,kind,extra
0,road_length_m,/elements/length/groupBy/boundary,snapshot,{}
1,road_element_count,/elements/count/groupBy/boundary,snapshot,{}
2,contributors_count,/users/count/groupBy/boundary,interval,{}
3,contributions_count,/contributions/count/groupBy/boundary,interval,{}
4,creation_contributions_count,/contributions/count/groupBy/boundary,interval,{'contributionType': 'creation'}
5,deletion_contributions_count,/contributions/count/groupBy/boundary,interval,{'contributionType': 'deletion'}
6,tag_change_contributions_count,/contributions/count/groupBy/boundary,interval,{'contributionType': 'tagChange'}
7,geometry_change_contributions_count,/contributions/count/groupBy/boundary,interval,{'contributionType': 'geometryChange'}


## 4. Response validation, checkpointing, retry/backoff, and adaptive splitting

In [5]:
def validate_temporal_body(body, expected_ids, kind):
    if not isinstance(body, dict):
        raise ValueError('Response body is not a JSON object.')
    groups = body.get('groupByResult')
    if not isinstance(groups, list):
        raise ValueError('Missing groupByResult list.')
    returned = [str(g.get('groupByObject')) for g in groups]
    if len(returned) != len(set(returned)):
        raise ValueError('Duplicate groupByObject values.')
    if set(returned) != set(expected_ids):
        raise ValueError(f'Boundary mismatch: missing={set(expected_ids)-set(returned)}, extra={set(returned)-set(expected_ids)}')
    for group in groups:
        result = group.get('result')
        if not isinstance(result, list) or not result:
            raise ValueError(f'Empty temporal result for {group.get("groupByObject")}.')
        for item in result:
            if 'value' not in item:
                raise ValueError('Temporal result item is missing value.')
            if kind == 'snapshot' and 'timestamp' not in item:
                raise ValueError('Snapshot item is missing timestamp.')
            if kind == 'interval' and not {'fromTimestamp','toTimestamp'}.issubset(item):
                raise ValueError('Interval item is missing fromTimestamp/toTimestamp.')
    return True

def checkpoint_path(metric, ids):
    token = f"{ids[0]}__{ids[-1]}__n{len(ids):03d}"
    return RAW_DIR / metric / f'{token}.json'

def post_request(metric, cfg, ids):
    path = checkpoint_path(metric, ids)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        cached = json.loads(path.read_text(encoding='utf-8'))
        body = cached.get('response', {}).get('body')
        if cached.get('response', {}).get('status_code') == 200:
            validate_temporal_body(body, ids, cfg['kind'])
            return body, 'cache'

    data = {
        'bpolys': json.dumps(batch_feature_collection(ids), ensure_ascii=False, separators=(',',':')),
        'filter': ROAD_FILTER,
        'format': 'json',
        'time': time_spec,
        **cfg['extra'],
    }
    url = f"{API_BASE}{cfg['endpoint']}"
    for attempt in range(1, MAX_ATTEMPTS + 1):
        response = session.post(url, data=data, timeout=REQUEST_TIMEOUT)
        try:
            body = response.json()
        except Exception:
            body = {'raw_text':response.text}
        envelope = {
            'request':{'metric':metric,'endpoint':cfg['endpoint'],'ids':ids,'filter':ROAD_FILTER,'time':time_spec,'extra':cfg['extra']},
            'response':{'status_code':response.status_code,'content_type':response.headers.get('content-type',''),'body':body},
            'recorded_at_utc':datetime.now(timezone.utc).isoformat(),
        }
        if response.ok:
            validate_temporal_body(body, ids, cfg['kind'])
            path.write_text(json.dumps(envelope, indent=2, ensure_ascii=False), encoding='utf-8')
            return body, 'request'
        error_path = path.with_suffix('.error.json')
        error_path.write_text(json.dumps(envelope, indent=2, ensure_ascii=False), encoding='utf-8')
        if response.status_code == 413:
            raise OverflowError('HTTP 413')
        if response.status_code not in TRANSIENT_STATUS or attempt == MAX_ATTEMPTS:
            raise RuntimeError(f"{metric} HTTP {response.status_code}: {json.dumps(body, ensure_ascii=False)[:1000]}")
        retry_after = response.headers.get('Retry-After')
        delay = float(retry_after) if retry_after and retry_after.isdigit() else min(2 ** attempt, 30)
        time.sleep(delay)
    raise RuntimeError('Request exhausted retries.')

def request_adaptive(metric, cfg, ids, depth=0):
    try:
        body, source = post_request(metric, cfg, ids)
        return [(ids, body, source)]
    except OverflowError:
        if len(ids) == 1:
            raise RuntimeError(f'{metric}: a single-cell request still returned HTTP 413.')
        middle = len(ids) // 2
        left = request_adaptive(metric, cfg, ids[:middle], depth+1)
        right = request_adaptive(metric, cfg, ids[middle:], depth+1)
        return left + right

def parse_temporal_body(body, metric, kind):
    rows = []
    for group in body['groupByResult']:
        gid = str(group['groupByObject'])
        for item in group['result']:
            if kind == 'snapshot':
                rows.append({'grid_1km_id':gid,'time':item['timestamp'],metric:float(item['value'])})
            else:
                rows.append({'grid_1km_id':gid,'from_time':item['fromTimestamp'],'to_time':item['toTimestamp'],metric:float(item['value'])})
    return pd.DataFrame(rows)

## 5. Execute or resume all monthly metric requests

In [6]:
metric_frames = {}
run_stats = []
for metric, cfg in METRICS.items():
    parts = []
    for batch_no, ids in enumerate(batches, start=1):
        returned_parts = request_adaptive(metric, cfg, ids)
        for actual_ids, body, source in returned_parts:
            parts.append(parse_temporal_body(body, metric, cfg['kind']))
            run_stats.append({'metric':metric,'planned_batch':batch_no,'cells':len(actual_ids),'source':source})
        print(f'{metric}: planned batch {batch_no:02d}/{len(batches)} — {len(ids)} cells')
    metric_frames[metric] = pd.concat(parts, ignore_index=True)

run_stats_df = pd.DataFrame(run_stats)
display(run_stats_df.groupby(['metric','source']).agg(request_units=('cells','size'), cells=('cells','sum')).reset_index())

road_length_m: planned batch 01/36 — 25 cells


road_length_m: planned batch 02/36 — 25 cells


road_length_m: planned batch 03/36 — 25 cells


road_length_m: planned batch 04/36 — 25 cells


road_length_m: planned batch 05/36 — 25 cells


road_length_m: planned batch 06/36 — 25 cells


... (279 outputs left out)

geometry_change_contributions_count: planned batch 34/36 — 25 cells


geometry_change_contributions_count: planned batch 35/36 — 25 cells


geometry_change_contributions_count: planned batch 36/36 — 20 cells


,metric,source,request_units,cells
0,contributions_count,request,36,895
1,contributors_count,request,36,895
2,creation_contributions_count,request,36,895
3,deletion_contributions_count,request,36,895
4,geometry_change_contributions_count,request,36,895
5,road_element_count,request,36,895
6,road_length_m,request,36,895
7,tag_change_contributions_count,request,36,895


## 6. Normalize monthly keys and validate each metric independently

In [7]:
metric_diagnostics = []
for metric, cfg in METRICS.items():
    frame = metric_frames[metric].copy()
    if cfg['kind'] == 'snapshot':
        frame['time'] = pd.to_datetime(frame['time'], utc=True)
        frame['month'] = frame['time'].dt.to_period('M').astype(str)
        key_cols = ['grid_1km_id','month']
    else:
        frame['from_time'] = pd.to_datetime(frame['from_time'], utc=True)
        frame['to_time'] = pd.to_datetime(frame['to_time'], utc=True)
        frame['month'] = frame['from_time'].dt.to_period('M').astype(str)
        key_cols = ['grid_1km_id','month']
    duplicate_keys = int(frame.duplicated(key_cols).sum())
    negative_values = int((frame[metric] < 0).sum())
    metric_diagnostics.append({
        'metric':metric,
        'rows':len(frame),
        'cells':frame['grid_1km_id'].nunique(),
        'months':frame['month'].nunique(),
        'duplicate_cell_month_keys':duplicate_keys,
        'null_values':int(frame[metric].isna().sum()),
        'negative_values':negative_values,
    })
    if frame['grid_1km_id'].nunique() != 895 or duplicate_keys or negative_values:
        raise ValueError(f'{metric} failed independent validation.')
    metric_frames[metric] = frame

metric_diagnostics_df = pd.DataFrame(metric_diagnostics)
display(metric_diagnostics_df)

/tmp/ipykernel_2145/1036092409.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame['month'] = frame['time'].dt.to_period('M').astype(str)
/tmp/ipykernel_2145/1036092409.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame['month'] = frame['time'].dt.to_period('M').astype(str)


/tmp/ipykernel_2145/1036092409.py:11: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame['month'] = frame['from_time'].dt.to_period('M').astype(str)
/tmp/ipykernel_2145/1036092409.py:11: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame['month'] = frame['from_time'].dt.to_period('M').astype(str)


/tmp/ipykernel_2145/1036092409.py:11: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame['month'] = frame['from_time'].dt.to_period('M').astype(str)
/tmp/ipykernel_2145/1036092409.py:11: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame['month'] = frame['from_time'].dt.to_period('M').astype(str)


/tmp/ipykernel_2145/1036092409.py:11: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame['month'] = frame['from_time'].dt.to_period('M').astype(str)
/tmp/ipykernel_2145/1036092409.py:11: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame['month'] = frame['from_time'].dt.to_period('M').astype(str)


,metric,rows,cells,months,duplicate_cell_month_keys,null_values,negative_values
0,road_length_m,201375,895,225,0,0,0
1,road_element_count,201375,895,225,0,0,0
2,contributors_count,200480,895,224,0,0,0
3,contributions_count,200480,895,224,0,0,0
4,creation_contributions_count,200480,895,224,0,0,0
5,deletion_contributions_count,200480,895,224,0,0,0
6,tag_change_contributions_count,200480,895,224,0,0,0
7,geometry_change_contributions_count,200480,895,224,0,0,0


## 7. Build the canonical cell-month panel

Activity metrics use the interval start month. State metrics use their snapshot month. The panel uses the intersection of months represented by both temporal semantics so every retained cell-month has a consistent basis.

In [8]:
snapshot_months = set.intersection(*[set(metric_frames[m]['month']) for m,cfg in METRICS.items() if cfg['kind']=='snapshot'])
interval_months = set.intersection(*[set(metric_frames[m]['month']) for m,cfg in METRICS.items() if cfg['kind']=='interval'])
panel_months = sorted(snapshot_months & interval_months)
if not panel_months:
    raise ValueError('No common months between state and activity metrics.')

panel = pd.MultiIndex.from_product([ordered_ids, panel_months], names=['grid_1km_id','month']).to_frame(index=False)
for metric, cfg in METRICS.items():
    cols = ['grid_1km_id','month',metric]
    panel = panel.merge(metric_frames[metric][cols], on=['grid_1km_id','month'], how='left', validate='one_to_one')

metric_cols = list(METRICS)
panel['road_length_km'] = panel['road_length_m'] / 1000.0
panel['road_density_km_per_km2'] = panel['road_length_km']
panel['active_mapping_month'] = panel['contributions_count'].gt(0)
panel['contributions_per_contributor'] = panel['contributions_count'].div(panel['contributors_count'].replace(0, pd.NA)).astype('Float64')
for col in ['creation_contributions_count','deletion_contributions_count','tag_change_contributions_count','geometry_change_contributions_count']:
    share_name = col.replace('_count','_share')
    panel[share_name] = panel[col].div(panel['contributions_count'].replace(0, pd.NA)).astype('Float64')

panel = panel.sort_values(['grid_1km_id','month']).reset_index(drop=True)
panel['road_length_delta_m'] = panel.groupby('grid_1km_id')['road_length_m'].diff()
panel['road_element_delta'] = panel.groupby('grid_1km_id')['road_element_count'].diff()
panel['cumulative_contributions'] = panel.groupby('grid_1km_id')['contributions_count'].cumsum()

print('Panel rows:', f'{len(panel):,}')
print('Cells:', panel['grid_1km_id'].nunique())
print('Months:', panel['month'].nunique(), panel['month'].min(), '→', panel['month'].max())
display(panel.head())

Panel rows: 200,480
Cells: 895
Months: 224 2007-11 → 2026-06


,grid_1km_id,month,road_length_m,road_element_count,contributors_count,contributions_count,creation_contributions_count,deletion_contributions_count,tag_change_contributions_count,geometry_change_contributions_count,...,road_density_km_per_km2,active_mapping_month,contributions_per_contributor,creation_contributions_share,deletion_contributions_share,tag_change_contributions_share,geometry_change_contributions_share,road_length_delta_m,road_element_delta,cumulative_contributions
0,1KME6094N8811,2007-11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,False,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,0.0
1,1KME6094N8811,2007-12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,False,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0
2,1KME6094N8811,2008-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,False,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0
3,1KME6094N8811,2008-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,False,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0
4,1KME6094N8811,2008-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,False,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0


## 8. Derive per-cell temporal summary indicators

In [9]:
def first_month_where(group, condition):
    values = group.loc[condition(group), 'month']
    return values.iloc[0] if len(values) else pd.NA

summary_rows = []
for gid, group in panel.groupby('grid_1km_id', sort=False):
    group = group.sort_values('month')
    active = group['contributions_count'].gt(0)
    summary_rows.append({
        'grid_1km_id':gid,
        'first_road_presence_month':first_month_where(group, lambda g: g['road_length_m'].gt(0)),
        'first_active_mapping_month':first_month_where(group, lambda g: g['contributions_count'].gt(0)),
        'last_active_mapping_month':group.loc[active, 'month'].iloc[-1] if active.any() else pd.NA,
        'active_months':int(active.sum()),
        'inactive_months':int((~active).sum()),
        'activity_persistence_ratio':float(active.mean()),
        'complete_month_contributions':float(group['contributions_count'].sum()),
        'peak_monthly_contributions':float(group['contributions_count'].max()),
        'mean_monthly_contributors':float(group['contributors_count'].mean()),
        'peak_monthly_contributors':float(group['contributors_count'].max()),
        'latest_complete_month_road_length_m':float(group['road_length_m'].iloc[-1]),
        'latest_complete_month_road_elements':float(group['road_element_count'].iloc[-1]),
        'positive_road_growth_months':int(group['road_length_delta_m'].gt(0).sum()),
        'negative_road_growth_months':int(group['road_length_delta_m'].lt(0).sum()),
        'median_monthly_contributions_when_active':float(group.loc[active,'contributions_count'].median()) if active.any() else 0.0,
    })

summary = pd.DataFrame(summary_rows)
display(summary.head())
display(summary.describe(include='all').T)

,grid_1km_id,first_road_presence_month,first_active_mapping_month,last_active_mapping_month,active_months,inactive_months,activity_persistence_ratio,complete_month_contributions,peak_monthly_contributions,mean_monthly_contributors,peak_monthly_contributors,latest_complete_month_road_length_m,latest_complete_month_road_elements,positive_road_growth_months,negative_road_growth_months,median_monthly_contributions_when_active
0,1KME6094N8811,2016-04,2016-03,2020-10,6,218,0.026786,26.0,15.0,0.031250,2.0,2200.95,3.0,2,0,3.0
1,1KME6094N8812,NaN,NaN,NaN,0,224,0.000000,0.0,0.0,0.000000,0.0,0.00,0.0,0,0,0.0
2,1KME6095N8810,2016-04,2016-03,2024-12,4,220,0.017857,22.0,9.0,0.022321,2.0,2482.74,4.0,1,1,5.5
3,1KME6095N8811,2016-04,2016-03,2024-12,5,219,0.022321,11.0,5.0,0.022321,1.0,1684.67,3.0,1,0,2.0
4,1KME6095N8812,2016-04,2016-03,2024-12,5,219,0.022321,11.0,5.0,0.022321,1.0,1828.31,3.0,1,0,2.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
grid_1km_id,895,895,1KME6094N8811,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
first_road_presence_month,793,56,2009-03,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
first_active_mapping_month,795,56,2009-02,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_active_mapping_month,795,76,2024-12,71,NaN,NaN,NaN,NaN,NaN,NaN,NaN
active_months,895.0,NaN,NaN,NaN,13.237989,13.590459,0.0,4.0,8.0,18.0,73.0
inactive_months,895.0,NaN,NaN,NaN,210.762011,13.590459,151.0,206.0,216.0,220.0,224.0
activity_persistence_ratio,895.0,NaN,NaN,NaN,0.059098,0.060672,0.0,0.017857,0.035714,0.080357,0.325893
complete_month_contributions,895.0,NaN,NaN,NaN,90.814525,178.748462,0.0,7.0,24.0,83.5,1802.0
peak_monthly_contributions,895.0,NaN,NaN,NaN,20.958659,33.023168,0.0,3.0,9.0,23.0,279.0
mean_monthly_contributors,895.0,NaN,NaN,NaN,0.080961,0.093985,0.0,0.017857,0.040179,0.102679,0.526786


## 9. Final integrity checks

In [10]:
expected_panel_rows = 895 * len(panel_months)
final_checks = {
    'panel_rows_complete': len(panel) == expected_panel_rows,
    'panel_cell_month_unique': not panel.duplicated(['grid_1km_id','month']).any(),
    'panel_895_cells': panel['grid_1km_id'].nunique() == 895,
    'panel_ids_match_grid': set(panel['grid_1km_id']) == set(ordered_ids),
    'all_primary_metrics_non_null': not panel[metric_cols].isna().any().any(),
    'all_primary_metrics_nonnegative': bool((panel[metric_cols] >= 0).all().all()),
    'summary_895_rows': len(summary) == 895,
    'summary_ids_unique': summary['grid_1km_id'].is_unique,
}
diagnostics = pd.DataFrame([{'check':k,'status':'PASS' if v else 'FAIL'} for k,v in final_checks.items()])
display(diagnostics)
if not all(final_checks.values()):
    raise ValueError(f'Final Notebook 06 validation failed: {final_checks}')

,check,status
0,panel_rows_complete,PASS
1,panel_cell_month_unique,PASS
2,panel_895_cells,PASS
3,panel_ids_match_grid,PASS
4,all_primary_metrics_non_null,PASS
5,all_primary_metrics_nonnegative,PASS
6,summary_895_rows,PASS
7,summary_ids_unique,PASS


## 10. Save reproducible Parquet and GeoParquet outputs

In [11]:
panel['api_version'] = api_version
panel['oshdb_from_timestamp'] = oshdb_start.isoformat()
panel['oshdb_to_timestamp'] = oshdb_end.isoformat()
panel['query_filter'] = ROAD_FILTER
panel['temporal_period'] = PERIOD
panel.to_parquet(OUTPUT_PANEL, index=False)
summary.to_parquet(OUTPUT_SUMMARY, index=False)

summary_geo = grid.merge(summary, on='grid_1km_id', how='left', validate='one_to_one')
summary_geo.to_parquet(OUTPUT_SUMMARY_GEO, index=False)

manifest = {
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'api_version':api_version,
    'oshdb_from_timestamp':oshdb_start.isoformat(),
    'oshdb_to_timestamp':oshdb_end.isoformat(),
    'complete_month_query':time_spec,
    'filter':ROAD_FILTER,
    'cells':895,
    'panel_months':len(panel_months),
    'panel_rows':len(panel),
    'metrics':METRICS,
    'outputs':[str(OUTPUT_PANEL.relative_to(PROJECT_ROOT)),str(OUTPUT_SUMMARY.relative_to(PROJECT_ROOT)),str(OUTPUT_SUMMARY_GEO.relative_to(PROJECT_ROOT))],
}
MANIFEST_FILE.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

print('Saved:', OUTPUT_PANEL)
print('Saved:', OUTPUT_SUMMARY)
print('Saved:', OUTPUT_SUMMARY_GEO)
print('Saved:', MANIFEST_FILE)

Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/processed/Petropolis/petropolis_ohsome_monthly_timeseries_895_cells.parquet
Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/processed/Petropolis/petropolis_ohsome_temporal_summary_895_cells.parquet
Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/processed/Petropolis/petropolis_grid_1km_with_temporal_summary.parquet
Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/raw/ohsome/Petropolis/monthly_timeseries_895_cells/manifest.json


## 11. Interpretation guardrails

- Road length and road-element count are monthly **states**, not counts of edits.
- Contributors and contributions are monthly **activity intervals**.
- Contribution-type counts can overlap; their shares are diagnostic ratios, not compositional shares that must sum to one.
- The first and last partial OSHDB months are excluded from the canonical panel so every activity observation represents a complete calendar month.
- `road_length_delta_m` is a change in mapped network state between snapshots; it is not equivalent to `geometryChange` contributions.
- Notebook 07 should use this panel to derive maturity, growth, persistence, saturation and contributor-activity indicators without re-querying the API.